# Training Run Log


## Sprint 6 — First real training run

Historical results from the first real (GPU, local machine) training run — moved here from
`notebooks/06_first_real_training_run.ipynb`, which keeps the rationale (GPU discovery,
MLflow/TensorBoard setup, AMP, checkpoint resume, config changes, stratified subsetting) and the
Findings/Next Steps analysis, which refers to this data by number.

### Results — Flower (completed run)

Run: `python scripts/train_baseline.py --config configs/flower.yaml --subset-per-class 25 --epochs 6`
(~25 images/class × 102 classes ≈ 2,550 training images, batch_size=16, AMP on)

Pulled directly from the MLflow/TensorBoard logs below — not retyped by hand.


In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("visionlab")
runs = client.search_runs([experiment.experiment_id], order_by=["start_time ASC"])

# Filter out the tracking-fix verification run (params={"foo": "bar"}) from Sprint 6, section 2.
real_runs = [r for r in runs if "model" in r.data.params]

for run in real_runs:
    print(f"dataset={run.data.params['dataset']:<10} status={run.info.status:<10} "
          f"batch_size={run.data.params['batch_size']} lr={run.data.params['lr']} "
          f"augmentation={run.data.params['augmentation_preset']:<8} "
          f"class_weighted={run.data.params['class_weighted_loss']}")
    print(f"  final: {run.data.metrics}")


Epoch-by-epoch curve, read from the TensorBoard event file
(`runs/flower_resnet50/`):

| Epoch | train_loss | val_loss | train_acc | val_acc |
|---|---|---|---|---|
| 1 | 3.6136 | 2.3582 | 0.2298 | 0.4132 |
| 2 | 1.1184 | 0.8027 | 0.7035 | 0.7885 |
| 3 | 0.4548 | 0.4694 | 0.8765 | 0.8655 |
| 4 | 0.2383 | 0.3499 | 0.9314 | 0.9108 |
| 5 | 0.1676 | 0.2885 | 0.9596 | 0.9218 |
| 6 | 0.1097 | 0.2941 | 0.9737 | 0.9230 |

**Reading this:** both losses drop sharply in the first 2-3 epochs (expected
— the pretrained ImageNet backbone already "knows" useful features, only
the new classifier head + fine-tuning need to catch up) and validation
accuracy tracks training accuracy closely through epoch 5, which is a good
sign for a 6-epoch run on ~2,550 images. One thing worth flagging, not
fixing yet: `val_loss` ticks *up* slightly in the last epoch (0.2885 →
0.2941) while `train_loss` keeps dropping — the earliest, mildest possible
signal of the model starting to overfit the small subset. Not a concern at
6 epochs, but worth watching if this same setup runs for many more epochs
on the full flower dataset.

### Results — Mushroom (completed run)

Run: `python scripts/train_baseline.py --config configs/mushroom.yaml --subset-per-class 25 --epochs 6`
(~25 images/class × 169 classes ≈ 4,225 training images, `augmentation_preset: light`, no class weighting)


In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

files = sorted(glob.glob("runs/mushroom_resnet50/events.out.tfevents.*"))
for f in files:
    ea = EventAccumulator(f)
    ea.Reload()
    for tag in ea.Tags().get("scalars", []):
        values = [(e.step + 1, round(e.value, 4)) for e in ea.Scalars(tag)]
        print(f"{tag:<15} {values}")


| Epoch | train_loss | val_loss | train_acc | val_acc |
|---|---|---|---|---|
| 1 | 4.5234 | 2.7997 | 0.1112 | 0.3587 |
| 2 | 2.3008 | 1.8899 | 0.4566 | 0.5042 |
| 3 | 1.3081 | 1.5916 | 0.6764 | 0.5662 |
| 4 | 0.7884 | 1.5100 | 0.8161 | 0.5831 |
| 5 | 0.4536 | 1.4446 | 0.9015 | 0.6034 |
| 6 | 0.2815 | 1.4031 | 0.9472 | 0.6197 |

**Reading this — and comparing to flower:** the gap between train_acc
(94.7%) and val_acc (62.0%) by epoch 6 is much wider than flower's
(97.4% vs. 92.3%) — mushroom is visibly overfitting this subset, flower
isn't (yet). It's tempting to list "169 vs. 102 classes" and "`light` vs.
`heavy` augmentation" as two independent, equally-weighted causes next to
a third about data scarcity. That would be misleading — there's one
primary cause here, and the other two are downstream of it, not
independent of it.

**Primary cause: this run gives mushroom an artificially scarce slice of
itself.** 25 images/class is 0.61% of mushroom's real per-class count
(4,080) — for flower, the same 25/class is close to its *actual* average
(~64/class, 38.92% of the full train set). Asking a
169-class problem to generalize from 25 examples/class is a fundamentally
harder few-shot-like task, independent of anything else in the config.

**Why the other two factors are secondary, not independent, causes:**

- **169 vs. 102 classes at equal per-class count** compounds the scarcity
  effect (more classes to separate from the same tiny per-class sample) —
  it doesn't explain overfitting on its own; it's a multiplier on the
  scarcity problem above.
- **`augmentation_preset: light` vs. `heavy` is a config decision that was
  correct for a different scenario.** Sprint 1/3 set `light` for mushroom
  *because* the real dataset has 4,080 images/class — abundant data, so
  aggressive augmentation wasn't judged necessary, and there was also an
  open concern about possible near-duplicates making heavier augmentation
  risky. Neither justification holds in a 25-images/class subset: here
  there's no abundance to lean on, so `light` provides less regularization
  than this specific (artificial) scenario would benefit from. That's a
  mismatch between the subset and the assumption the config decision was
  based on — not evidence that the decision itself was wrong for the real,
  4,080-images/class dataset.

**The conclusion this section does *not* support:** "mushroom's config
needs to change." This overfitting is the expected result of solving a
169-class problem with an artificially scarce 25-images/class slice — not
a pipeline, model, or config defect. Changing `augmentation_preset` (or
anything else) based on this smoke test, before ever training on the real
4,080-images/class data, would be reacting to an artifact of the test
setup rather than a real signal.


## Run 1 — Mushroom baseline (job 6061165)

- Subset: 300 images/class (169 classes ≈ 50,700 train), full val (15,616)
- 6 epochs, CPU-only (`orfoz`, 56 cores)
- Metrics available: loss/accuracy only — no macro-F1, no per-class breakdown
- Result: train_acc 95.4%, best val_acc 80.2% at epoch 3, final val_acc 78.0% (epoch 6) —
  val_loss rises after epoch 3 while train_loss keeps falling, i.e. overfitting past that point
- Duration: 3h11m47s
- Checkpoint: `outputs/checkpoints/mushroom_resnet50.pt` — last-epoch weights, not the best-val
  epoch (checkpoint saving isn't best-epoch-aware yet)


## What changed before Run 2

- `src/evaluation/metrics.py` (new): macro-F1, per-class precision/recall/F1, top-10 most
  confused class pairs, worst-10 classes by F1
- `src/training/engine.py`: `evaluate()` now also returns `y_true`/`y_pred` so the above can be
  computed without a second pass over the validation set
- `src/training/tracking.py`: `log_epoch_metrics` accepts extra per-epoch metrics
  (`val_macro_f1`, `epoch_duration_seconds`); added `log_run_duration` and
  `log_evaluation_report` (MLflow)
- `scripts/train_baseline.py`: logs macro-F1 and epoch duration every epoch (MLflow +
  TensorBoard); after the last epoch, prints and saves a JSON report — worst-10 classes,
  top-10 confused pairs — to `outputs/reports/{dataset}_{model}_eval_report.json`, and logs
  total run duration to MLflow
- `pyproject.toml`: added `scikit-learn` as an explicit dependency (was only a transitive one
  via mlflow before)
- Validated with two smoke-test jobs (50 and 200 image subsets, with and without MLflow) before
  submitting real runs


## Run 2 — completed (mushroom job 6066357, flower job 6066358)

New artifacts per run: `outputs/reports/{dataset}_{model}_eval_report.json`, plus MLflow
`run_duration_seconds` and an `evaluation_report.json` artifact.

### Mushroom (300/class subset, same as Run 1, now with the new metrics)

- Duration: 13081.5s (3h38m), 56 CPU cores (`orfoz274`)
- train_acc 95.2%, val_acc 80.2%, **macro-F1 0.7596** (final epoch)
- Unlike Run 1, val_acc keeps improving through epoch 6 instead of peaking at epoch 3 — less
  overfitting this time (run-to-run variance from weight init / data shuffling, same subset size)
- Worst-10 classes are mostly lichens, not true mushrooms (Fomitopsis mounceae F1 0.385,
  Boletus reticulatus 0.395, Phaeophyscia orbicularis 0.451, …) — support 35-83 images each
- Top confused pair: Xanthoria parietina → Vulpicida pinastri (66 misclassifications) — both
  lichens, visually similar; several Fomitopsis/Fomes pairs also recur

### Flower (full dataset: 6,552 train / 818 val, first real run with macro-F1)

- Duration: 1451.0s (24m), 56 CPU cores (`orfoz286`)
- train_acc 97.8%, val_acc 97.8%, **macro-F1 0.9726** (final epoch) — strong result on the full
  dataset, consistent with Sprint 4's EfficientNet research note (up to 98.8% reported achievable)
- Worst-10 classes are almost entirely low-support ones (canterbury bells support=2, monkshood
  support=3, …) — F1 noise from small sample size, not a systematic weakness
- Top confused pairs are all single-count, isolated misclassifications — no dataset-wide
  confusion pattern like mushroom's lichen pairs


## What changed before Run 3

**Tracking: MLflow/TensorBoard → Weights & Biases.** Both replaced by a single W&B integration
(`src/training/tracking.py` rewritten, `scripts/train_baseline.py`'s `--no-mlflow`/
`--no-tensorboard` collapsed into one `--no-wandb`). `pyproject.toml` swaps `mlflow`/
`tensorboard` for `wandb`. Old local artifacts (`mlflow.db`) deleted; `runs/` (TensorBoard)
left on disk but no longer written to. Reasoning: wanted per-run names + config + notes in one
dashboard instead of cross-referencing this notebook for what a run's hyperparameters were.

**Run-name bug fix.** Run names are now `{dataset}_{model}_{subset_label}_ep{epochs}_{job_id}`.
The subset label previously only checked `--subset-per-class` and silently called anything else
(including a `--limit`-truncated smoke test) `"full"` — fixed to check `--limit` too
(`limitN` label). Actual `train_dataset`/`val_dataset` sizes are now also logged as W&B config
so a run's real scale never has to be inferred from its name.

**Best-checkpoint tracking.** `save_checkpoint` now runs twice per improving epoch: the existing
resumable "latest" checkpoint (`{dataset}_{model}.pt`, includes optimizer/scaler state, for
`--resume`), and a new `{dataset}_{model}_best.pt` (model weights only) written whenever
`val_loss` hits a new low. The final evaluation report (macro-F1, worst classes, confused pairs)
now uses the *best* epoch's predictions, not the last one — Run 1/2 reported the last epoch's
numbers even when an earlier epoch generalized better.

**Early stopping + LR scheduling.** New `training.early_stopping_patience` (default 5) and
`training.lr_scheduler_patience` (default 2) config keys. Training stops once `val_loss` hasn't
improved for `early_stopping_patience` epochs; `torch.optim.lr_scheduler.ReduceLROnPlateau`
halves the learning rate after `lr_scheduler_patience` epochs of no improvement. Both configs'
`epochs` raised to 30 — a ceiling, not a target; early stopping decides the real stop.

**Gradient clipping.** `train_one_epoch` clips gradients to `max_norm=1.0` before every optimizer
step (cheap stability guard, hardcoded rather than config-exposed since it's not something we're
tuning per dataset).

**Config cleanup.** `pin_memory: true → false` in both configs — it only helps on CUDA and only
ever warned on CPU (confirmed: no GPU access is being used, per earlier decision).
`mixed_precision: true` left as-is; it's already a no-op on CPU (`use_amp` requires
`device.type == "cuda"`), so nothing to fix there despite it looking related.

**Validation before the real run.** A smoke test on a 50-images/class mushroom subset confirmed
the run-name fix and best-checkpoint creation, but an initial 20-minute time budget was too
short to see early stopping actually fire (only 1 epoch completed, ~13-20 min/epoch at this
scale). Rather than burn more debug-partition time extending that test, moved straight to the
real runs below — early stopping, LR scheduling, and best-checkpoint selection all ended up
getting fully exercised there anyway (see Flower results).


## Run 3 — mushroom job 6072512, flower job 6072513 (both completed)

Both submitted with `--epochs 30` (ceiling) via `scripts/train_truba_cpu.sbatch`, `orfoz`
partition: `sbatch scripts/train_truba_cpu.sbatch configs/mushroom.yaml 30 "" 300` (300/class
subset) and `sbatch scripts/train_truba_cpu.sbatch configs/flower.yaml 30` (full dataset).


### Flower (full dataset) — completed

- **Early stopping fired at epoch 18** (val_loss hadn't improved for 5 epochs) — didn't run the
  full 30-epoch ceiling
- **LR scheduler fired twice**: 1e-4 → 5e-5 at epoch 10, → 2.5e-5 at epoch 16, both right after
  multi-epoch val_loss plateaus
- **Best epoch: 13** (val_loss=0.0680), correctly *not* the last epoch (18) — this is exactly the
  scenario best-checkpoint tracking was added for
- train_acc 99.98% (epoch 18), val_acc 98.8% (best epoch), **macro-F1 0.9882** (best epoch) — up
  from Run 2's 0.9726, from more epochs + LR decay + reporting the best epoch instead of the last
- Duration: 4069.9s (68 min) — longer than Run 2 (24 min) because it ran 18 epochs instead of a
  fixed 6, not because anything got slower per-epoch (~220s/epoch, same as before)
- W&B run: `flower_resnet50_full_ep30_6072513`

Worst-10 classes (by F1) — all low-support, not a systematic weakness:

| label | precision | recall | f1 | support |
|---|---|---|---|---|
| lenten rose | 0.714 | 1.000 | 0.833 | 5 |
| camellia | 0.889 | 0.800 | 0.842 | 10 |
| garden phlox | 0.750 | 1.000 | 0.857 | 3 |
| mallow | 0.857 | 0.857 | 0.857 | 7 |
| ball moss | 1.000 | 0.833 | 0.909 | 6 |
| tree mallow | 0.833 | 1.000 | 0.909 | 5 |
| foxglove | 0.929 | 0.929 | 0.929 | 14 |
| cape flower | 0.889 | 1.000 | 0.941 | 8 |
| hippeastrum | 1.000 | 0.889 | 0.941 | 9 |
| columbine | 1.000 | 0.900 | 0.947 | 10 |

Top-10 confused pairs — every single one is a single isolated misclassification (count=1), no
dataset-wide confusion pattern like mushroom's recurring lichen pairs:

ball moss→foxglove, camellia→mallow, camellia→snapdragon, clematis→lenten rose,
columbine→lenten rose, foxglove→cyclamen, hibiscus→tree mallow, hippeastrum→cape flower,
mallow→camellia, morning glory→garden phlox


![Flower Run 3 metrics](assets/flower_run3_metrics.png)


### Mushroom (300/class subset) — completed

- **Early stopping fired at epoch 7** (val_loss hadn't improved for 5 epochs) — stopped well
  short of the 30-epoch ceiling
- **LR scheduler fired once**: 1e-4 → 5e-5 at epoch 5, after val_loss plateaued past epoch 2
- **Best epoch (by val_loss): 2** (val_loss=0.7664) — but this is *not* the epoch with the
  highest macro-F1. Epoch 7's macro-F1 (0.7865) is clearly higher than epoch 2's (0.7315), even
  though epoch 7's val_loss (0.8119) is worse. The model kept getting more accurate/higher-F1
  after epoch 2 while val_loss crept up — likely growing overconfidence on wrong predictions
  (higher-magnitude logits push loss up even as more predictions land correctly). Worth
  discussing: val_loss may not be the right signal to select the "best" checkpoint on for this
  dataset — macro-F1 or val_accuracy might track what we actually care about better. Not changed
  yet, flagging for a decision.
- train_acc 98.2% (epoch 7), val_acc 82.5% (epoch 7) / 78.1% (epoch 2, the saved-best checkpoint)
- **macro-F1 0.7315** (best-by-val_loss epoch 2) — reported by the current best-checkpoint logic;
  epoch 7 alone would have scored 0.7865
- Worst-10 classes: mostly lichens again (Volvopluteus gloiocephalus F1 0.301, Physcia
  adscendens 0.400, Boletus reticulatus 0.432, …) — same pattern as Run 1/2
- Top confused pair: Fomitopsis pinicola → Fomes fomentarius (51 misclassifications); several
  Parmelia/Physcia/Hypogymnia lichen pairs also recur, consistent with Run 1/2's lichen-confusion
  pattern
- Duration: 12395.7s (3h27m) — 7 epochs instead of Run 2's fixed 6, but early-stopped rather than
  running to a high ceiling
- W&B run: `mushroom_resnet50_subset300_ep30_6072512`


![Mushroom Run 3 metrics](assets/mushroom_run3_metrics.png)


## What changed before Run 4

**Early stopping / best-checkpoint criterion: val_loss → val_macro_f1.** Run 3's mushroom job
picked epoch 2 as "best" (lowest val_loss=0.7664, macro-F1 0.7315) even though epoch 7 had a
clearly higher macro-F1 (0.7865) with a worse val_loss — the model kept getting more accurate
while val_loss crept up (likely growing prediction confidence, not just correctness). Since we
actually care about accuracy/F1, not loss, both early stopping and best-checkpoint selection in
`scripts/train_baseline.py` now track `val_macro_f1` (higher is better) instead of `val_loss`.
The LR scheduler (`ReduceLROnPlateau`) was left on `val_loss` — a finer-grained, more continuous
signal is arguably still the right thing for LR decay even though it's the wrong thing for
"should we keep training at all."

**Mushroom `early_stopping_patience`: 5 → 8.** Run 3 stopped at epoch 7 with macro-F1 still
climbing; more patience gives it room to actually plateau before quitting. Flower's patience (5)
was left unchanged — it already showed a clean plateau in Run 3.

**Validated with a smoke test** (`--limit 200`, 4 epochs, `--no-wandb`) before relaunching —
confirmed the best-checkpoint now tracks the highest-macro-F1 epoch, no crashes.


## Run 4 — mushroom job 6077989 (manually stopped at epoch 20/30)

Same command as Run 3 (`sbatch scripts/train_truba_cpu.sbatch configs/mushroom.yaml 30 "" 300`),
just with the criterion/patience changes above.

**Unexpected slowdown — node contention.** Epochs took 2900-4760s this run vs. Run 3's
1720-1860s for the *same* subset/model — up to 2.5x slower. `scontrol show node orfoz298` showed
`CPUAlloc=61` while we only requested 56 — another job was sharing the same physical node,
competing for cache/memory bandwidth. `--cpus-per-task=56` reserves cores but not the whole node;
`--exclusive` would avoid this on future submissions.

**Manually cancelled at epoch 20 — clear overfitting plateau, not a bug.** By epoch 18
(best epoch), train_acc had saturated (99.9%, train_loss≈0.002) while val_macro_f1 had been flat
since ~epoch 10 (0.80-0.81, noisy, no clear upward trend) — see the dashed line in the chart
below. `early_stopping_patience=8` meant it would have kept running until ~epoch 26 or the
epoch-30 ceiling before stopping itself, but at ~50-80 min/epoch under the node contention above,
that's another 6-10+ hours for a model that had already stopped improving. The epoch-18
best-checkpoint was safely saved regardless, so cancelling costs nothing but time.

Since the job never reached its normal end-of-training block, the evaluation report (worst-10
classes, confused pairs) was regenerated afterward by loading the epoch-18 best checkpoint and
re-running evaluation on the full validation set (`scripts/train_baseline.py`'s reporting logic,
run standalone).

- **Best epoch: 18** (val_macro_f1=0.8126, val_acc=84.80%) — up from Run 3's 0.7315/78.1%
  (comparing like-for-like: Run 3's *actual* peak was epoch 7's 0.7865/82.7%, so the real gain
  from more patience is smaller than the headline number suggests, but still a genuine
  improvement, plus we're now confidently capturing the true peak instead of an early one)
- train_acc 99.9%, val_acc 84.8% — a ~15-point train/val gap, the overfitting signature above
- W&B run: `mushroom_resnet50_subset300_ep30_6077989`

Worst-10 classes (by F1) — mostly true mushrooms this time, not lichens like Run 3:

| label | precision | recall | f1 | support |
|---|---|---|---|---|
| Fomitopsis mounceae | 0.299 | 0.725 | 0.423 | 40 |
| Leccinum aurantiacum | 0.430 | 0.482 | 0.455 | 83 |
| Boletus reticulatus | 0.458 | 0.579 | 0.512 | 38 |
| Leccinum versipelle | 0.535 | 0.585 | 0.559 | 65 |
| Amanita persicina | 0.467 | 0.700 | 0.560 | 40 |
| Trametes ochracea | 0.562 | 0.675 | 0.614 | 40 |
| Phellinus igniarius | 0.469 | 0.900 | 0.616 | 50 |
| Galerina marginata | 0.568 | 0.714 | 0.633 | 35 |
| Suillus granulatus | 0.683 | 0.603 | 0.641 | 68 |
| Physcia adscendens | 0.557 | 0.800 | 0.657 | 55 |

Top-10 confused pairs — genus-level confusion dominates (Fomitopsis/Fomes bracket fungi,
Leccinum boletes), plus the same recurring lichen pairs from Run 1-3:

| true → predicted | count |
|---|---|
| Fomitopsis pinicola → Fomitopsis mounceae | 54 |
| Xanthoria parietina → Vulpicida pinastri | 30 |
| Fomes fomentarius → Fomitopsis betulina | 28 |
| Amanita muscaria → Amanita persicina | 24 |
| Fomes fomentarius → Phellinus igniarius | 24 |
| Fomes fomentarius → Ganoderma applanatum | 22 |
| Hypogymnia physodes → Parmelia sulcata | 21 |
| Evernia prunastri → Evernia mesomorpha | 18 |
| Leccinum scabrum → Leccinum aurantiacum | 18 |
| Pleurotus pulmonarius → Pleurotus ostreatus | 18 |

**Reading this:** `Fomitopsis pinicola → Fomitopsis mounceae` alone accounts for 54 of the
val set's errors — same genus, closely related bracket fungi. Combined with `Fomes fomentarius`'s
three separate confusions (→ Fomitopsis betulina, → Phellinus igniarius, → Ganoderma
applanatum — all polypore/bracket fungi) and the Leccinum/Amanita/Pleurotus species-pairs, the
error pattern is consistent across Run 1-4: **the model confuses closely related species within
the same genus, not unrelated ones.** This looks like a genuine visual-similarity ceiling for a
300/class subset, not a pipeline defect — worth checking visually (next step) before concluding
whether more data would resolve it or whether these pairs are inherently hard even with more
examples.


![Mushroom Run 4 metrics](assets/mushroom_run4_metrics.png)
